```
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
```

# RMI Sample Queries: RMI Planner (GA)

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/RMI_Planner_Samples.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Froads_management_insights%2Frmi_sample_queries%2Fnotebooks%2FRMI_Planner_Samples.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/RMI_Planner_Samples.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/RMI_Planner_Samples.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>



This notebook contains sample queries for the **RMI Planner** persona, specifically for the **GA** stage.

## 1. Setup

In [ ]:
from google.colab import auth
import pandas as pd

auth.authenticate_user()

project_id = 'your-project-id' #@param {type:"string"}

### Writable Dataset

Several queries in this notebook (e.g., those creating Materialized Views, Models, or Views) require a **writable dataset** within your own project. 
**Note**: The source `boston_oct_2025_sample_data` dataset is a read-only subscription and cannot be used to store new resources.

Run the cell below to create a new dataset (e.g., `rmi_analysis`) in your project if you haven't already.

**Important**: When running queries that create new BigQuery resources (e.g., tables, views, models) outside of these `%%bigquery` magic cells, remember to manually prepend the job ID with `msqlfactory--` for proper tracking. For example: `bq query --job_id=msqlfactory--your-descriptive-job-name ...`

In [ ]:
dataset_id = "rmi_analysis" #@param {type:"string"}
! bq --location=US mk --dataset {project_id}:{dataset_id}

## GA (General Availability)

### rmip1_usage_projection.sql
**Business Question**: Based on current data, what is the rate of record creation, and how will it scale?
Use Case: Helps sales teams estimate BigQuery storage and compute growth as a customer increases their monitoring footprint from a small pilot to an enterprise-wide fleet.
Product Stage: GA
Estimated Bytes Processed: < 1 MB (Standard SQL on RMI Tables)

In [ ]:
%%bigquery --project {project_id} df_rmip1_usage_projection
WITH daily_stats AS (
  SELECT
    DATE(record_time) as log_date,
    selected_route_id,
    COUNT(*) as records_per_day
  FROM `boston_oct_2025_sample_data.historical_travel_time`
  WHERE record_time BETWEEN '2025-10-01' AND '2025-11-01'
  GROUP BY 1, 2
),
avg_usage AS (
  SELECT
    AVG(records_per_day) as avg_daily_records_per_route
  FROM daily_stats
)
SELECT
  ROUND(avg_daily_records_per_route, 2) as avg_records_per_route_per_day,
  -- Parameter: Target fleet size (e.g., 5,000 routes)
  5000 as target_route_count,
  ROUND(avg_daily_records_per_route * 5000, 0) as estimated_total_daily_records,
  -- Extrapolate to monthly volume in millions of records
  ROUND(avg_daily_records_per_route * 5000 * 30 / 1000000, 2) as estimated_monthly_millions
FROM avg_usage;


### rmip2_customer_roi.sql
**Business Question**: How much total time is lost to congestion across different customer service tiers?
Use Case: Translates raw traffic data into "Business Value" by quantifying the potential time savings for priority routes, justifying the monitoring cost and providing a clear ROI for the customer.
Product Stage: GA
Estimated Bytes Processed: < 1 MB (Standard SQL on RMI Tables)

In [ ]:
%%bigquery --project {project_id} df_rmip2_customer_roi
SELECT
  JSON_EXTRACT_SCALAR(route_attributes, '$.tier') as service_tier,
  -- Aggregate total lost time (Actual - Free-flow) converted to hours
  ROUND(SUM(duration_in_seconds - static_duration_in_seconds) / 3600, 1) as total_delay_hours,
  COUNT(DISTINCT h.selected_route_id) as monitored_routes,
  -- Average performance multiplier
  ROUND(AVG(SAFE_DIVIDE(duration_in_seconds, static_duration_in_seconds)), 2) as avg_delay_index
FROM `boston_oct_2025_sample_data.historical_travel_time` h
JOIN `boston_oct_2025_sample_data.routes_status` s ON h.selected_route_id = s.selected_route_id
WHERE h.record_time BETWEEN '2025-10-01' AND '2025-11-01'
  -- Filter for records where actual was slower than free-flow
  AND (duration_in_seconds - static_duration_in_seconds) > 0
GROUP BY 1
HAVING service_tier IS NOT NULL
ORDER BY total_delay_hours DESC;


### rmip3_segment_estimation.sql
**Business Question**: How many physical road segments exist in our target study area, categorized by class?
Use Case: Helps sales and solution architects estimate the "Total Addressable Monitoring" footprint for a city, aiding in pricing and coverage strategy.
Product Stage: GA
Estimated Bytes Processed: ~1 MB (Uses BigQuery Public Dataset: Overture Maps)

In [ ]:
%%bigquery --project {project_id} df_rmip3_segment_estimation
/*
  EXTERNAL DEPENDENCY: 
  RMI monitoring is based on user-defined routes. To understand the underlying 
  physical scale of an area, this query joins with the Overture Maps public 
  dataset to provide a baseline count of all physical road segments.
*/

WITH target_boundary AS (
  SELECT geometry
  FROM `bigquery-public-data.overture_maps.division_area`
  WHERE names.primary = 'Boston' 
    AND country = 'US' 
    AND region = 'US-MA'
    AND class = 'land'
)
SELECT
  -- Group by physical road classification (e.g., motorway, primary, local)
  class as road_class,
  subtype,
  COUNT(*) as segment_count,
  ROUND(SUM(ST_LENGTH(s.geometry)) / 1000, 2) as total_length_km
FROM `bigquery-public-data.overture_maps.segment` s
JOIN target_boundary b ON ST_INTERSECTS(s.geometry, b.geometry)
WHERE s.subtype = 'road'
GROUP BY 1, 2
ORDER BY segment_count DESC;


### rmip4_area_boundary.sql
**Business Question**: How can I create a reusable, open-source administrative boundary for my target study area?
Use Case: Establishes a "Master Boundary" for a city or region using public data. This view can then be joined with RMI tables to automate geofencing and localized reporting.
Product Stage: GA
Estimated Bytes Processed: ~1 MB (Uses BigQuery Public Dataset: Overture Maps)

In [ ]:
%%bigquery --project {project_id} df_rmip4_area_boundary
/*
  NOTE: This query creates a persistent view of a target city's official boundary.
  The source dataset (e.g., `boston_oct_2025_sample_data`) is read-only.
  This view MUST be created in a separate, writable dataset within your project.
  Replace `your-project.your-dataset` with your target location.
*/

CREATE OR REPLACE VIEW `your-project.your-dataset.target_area_boundary` 
(
  division_id OPTIONS(description="Stable identifier for the administrative division."),
  area_name OPTIONS(description="The primary display name (e.g. Boston)."),
  region OPTIONS(description="The ISO state/province code (e.g. US-MA)."),
  country OPTIONS(description="The ISO country code."),
  geometry OPTIONS(description="The physical land boundary of the division as a GEOGRAPHY polygon.")
)
OPTIONS(
  description="A reusable administrative boundary for geofencing RMI analytical assets."
)
AS
SELECT 
  id AS division_id,
  names.primary AS area_name,
  region,
  country,
  geometry
FROM `bigquery-public-data.overture_maps.division_area`
WHERE names.primary = 'Boston' 
  AND country = 'US' 
  AND region = 'US-MA'
  AND class = 'land';
